# Deep Learning Route Choice Models

MNL 대비 딥러닝 경로 선택 모델 성능 비교.  
- **DNN**: Feedforward → utility → softmax (순수 예측)
- **TasteNet**: β = g(context) → V = Xβ(z) (해석 가능, OD 거리별 다른 β)
- **ResLogit**: V = Xβ_MNL + DNN(X) (MNL 잔차 학습)

In [ ]:
# Cell 1: Setup & Data Loading
import sys
import json
import numpy as np
import pandas as pd
import torch
from pathlib import Path

sys.path.insert(0, '.')
from module.data import (
    create_dataloaders, MODEL_FEATURES, FEATURE_LABELS,
    CONTEXT_FEATURES, MAX_ALTS,
)
from module.models import (
    DNNChoiceModel, TasteNetModel, ResLogitModel, load_mnl_beta,
)
from module.train import (
    train_model, evaluate_model, print_metrics, compute_ll_null,
)

DATA_DIR = '../../../data/training_set'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

# DataLoader 생성
train_loader, test_loader, train_ds, test_ds = create_dataloaders(
    data_dir=DATA_DIR, batch_size=2048,
)

# 검증: OD 수
print(f'\nExpected: train=478,345, test=119,587')
print(f'Actual:   train={len(train_ds):,}, test={len(test_ds):,}')

n_features = train_ds.n_features
n_context = train_ds.n_context
print(f'Features: {n_features}, Context: {n_context}')

In [ ]:
# Cell 2: LL(0) 검증
# MNL train LL(0) = -37,222,390.89

train_ll0 = compute_ll_null(train_ds)
test_ll0 = compute_ll_null(test_ds)

print(f'Train LL(0): {train_ll0:,.4f}')
print(f'Test  LL(0): {test_ll0:,.4f}')
print(f'MNL   LL(0): -37,222,390.8911 (reference)')
print(f'Diff:        {abs(train_ll0 - (-37222390.8911)):.4f}')

In [ ]:
# Cell 3: MNL baseline 결과 로드
with open(Path(DATA_DIR) / 'mnl_coefficients.json', 'r') as f:
    mnl_coeff = json.load(f)
with open(Path(DATA_DIR) / 'model_evaluation.json', 'r') as f:
    mnl_eval = json.load(f)

mnl_metrics = {
    'rho_sq': mnl_eval['test_stats']['rho_squared'],
    'fpr_top1': mnl_eval['test_stats']['top1_accuracy'],
    'fpr_top3': mnl_eval['test_stats']['top3_accuracy'],
    'rmse': mnl_eval['test_stats']['rmse'],
    'll_beta': mnl_eval['test_stats']['ll_beta'],
    'll_0': mnl_eval['test_stats']['ll_0'],
}

print('MNL Baseline:')
print(f'  ρ²:     {mnl_metrics["rho_sq"]:.4f}')
print(f'  FPR-1:  {mnl_metrics["fpr_top1"]:.4f}')
print(f'  FPR-3:  {mnl_metrics["fpr_top3"]:.4f}')
print(f'  RMSE:   {mnl_metrics["rmse"]:.4f}')

## Model 1: DNN

In [ ]:
# Cell 4: DNN 학습
print('=== DNN Model ===')
dnn_model = DNNChoiceModel(n_features, hidden_dims=(64, 32), dropout=0.1)
print(dnn_model)
n_params_dnn = sum(p.numel() for p in dnn_model.parameters())
print(f'Parameters: {n_params_dnn:,}')

dnn_history, dnn_state = train_model(
    dnn_model, train_loader, test_loader, train_ds, test_ds,
    lr=1e-3, epochs=100, patience=15, device=DEVICE,
)

In [ ]:
# Cell 5: DNN 평가
dnn_metrics, dnn_preds, dnn_targets, dnn_masks, dnn_weights = evaluate_model(
    dnn_model, test_loader, test_ds, device=DEVICE,
)
print_metrics('DNN', dnn_metrics)

## Model 2: TasteNet

In [ ]:
# Cell 6: TasteNet 학습
print('=== TasteNet Model ===')
tastenet_model = TasteNetModel(n_features, n_context, hidden_dim=32)
print(tastenet_model)
n_params_tn = sum(p.numel() for p in tastenet_model.parameters())
print(f'Parameters: {n_params_tn:,}')

tn_history, tn_state = train_model(
    tastenet_model, train_loader, test_loader, train_ds, test_ds,
    lr=1e-3, epochs=100, patience=15, device=DEVICE,
)

In [ ]:
# Cell 7: TasteNet 평가
tn_metrics, tn_preds, tn_targets, tn_masks, tn_weights = evaluate_model(
    tastenet_model, test_loader, test_ds, device=DEVICE,
)
print_metrics('TasteNet', tn_metrics)

## Model 3: ResLogit

In [ ]:
# Cell 8: ResLogit 학습
print('=== ResLogit Model ===')

# MNL β를 scaled space로 변환
mnl_beta_scaled = load_mnl_beta(
    Path(DATA_DIR) / 'mnl_coefficients.json',
    train_ds.scaler,
)
print(f'MNL β (scaled): {mnl_beta_scaled}')

reslogit_model = ResLogitModel(
    n_features, mnl_beta=mnl_beta_scaled,
    hidden_dim=32, dropout=0.1, freeze_mnl=False,
)
print(reslogit_model)
n_params_rl = sum(p.numel() for p in reslogit_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {n_params_rl:,}')

rl_history, rl_state = train_model(
    reslogit_model, train_loader, test_loader, train_ds, test_ds,
    lr=5e-4, epochs=100, patience=15, device=DEVICE,
)

In [ ]:
# Cell 9: ResLogit 평가
rl_metrics, rl_preds, rl_targets, rl_masks, rl_weights = evaluate_model(
    reslogit_model, test_loader, test_ds, device=DEVICE,
)
print_metrics('ResLogit', rl_metrics)

## Comparison & Interpretation

In [ ]:
# Cell 10: 모델 비교 테이블
results = {
    'MNL': mnl_metrics,
    'DNN': dnn_metrics,
    'TasteNet': tn_metrics,
    'ResLogit': rl_metrics,
}

print('=' * 75)
print('Model Comparison — Test Set')
print('=' * 75)
print(f'{"Metric":<20} {"MNL":>12} {"DNN":>12} {"TasteNet":>12} {"ResLogit":>12}')
print('-' * 75)

metrics_display = [
    ('McFadden ρ²', 'rho_sq', '.4f'),
    ('FPR Top-1', 'fpr_top1', '.4f'),
    ('FPR Top-3', 'fpr_top3', '.4f'),
    ('RMSE', 'rmse', '.4f'),
    ('LL(β)', 'll_beta', ',.0f'),
]

for label, key, fmt in metrics_display:
    vals = []
    for model_name in ['MNL', 'DNN', 'TasteNet', 'ResLogit']:
        v = results[model_name].get(key, float('nan'))
        vals.append(f'{v:{fmt}}')
    print(f'{label:<20} {vals[0]:>12} {vals[1]:>12} {vals[2]:>12} {vals[3]:>12}')

print('-' * 75)

# MNL 대비 개선율
print('\n=== MNL 대비 개선 ===')
for model_name in ['DNN', 'TasteNet', 'ResLogit']:
    m = results[model_name]
    rho_diff = m['rho_sq'] - mnl_metrics['rho_sq']
    fpr_diff = m['fpr_top1'] - mnl_metrics['fpr_top1']
    rmse_diff = m['rmse'] - mnl_metrics['rmse']
    print(f'  {model_name:<10}: Δρ²={rho_diff:+.4f}, ΔFPR-1={fpr_diff:+.4f}, ΔRMSE={rmse_diff:+.4f}')

In [ ]:
# Cell 11: TasteNet β 해석
import matplotlib.pyplot as plt

# Context 범위: OD distance 1~50km
od_dists = np.linspace(1, 50, 100)
cs_sizes = np.full(100, 3.0)  # choice set size 고정=3

# Scale context
z_raw = np.column_stack([od_dists, cs_sizes])
z_scaled = train_ds.context_scaler.transform(z_raw).astype(np.float32)
z_tensor = torch.from_numpy(z_scaled).to(DEVICE)

betas = tastenet_model.get_betas(z_tensor).cpu().numpy()

# 주요 피처의 β 변화 시각화
key_features = [
    (0, 'IVT (min)'),
    (5, 'Total dist (km)'),
    (6, 'Transfers'),
    (8, 'Has bus'),
    (9, 'Has train'),
]

fig, axes = plt.subplots(1, len(key_features), figsize=(4 * len(key_features), 3.5))
for ax, (idx, label) in zip(axes, key_features):
    ax.plot(od_dists, betas[:, idx], 'b-', linewidth=2)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.set_xlabel('OD Distance (km)')
    ax.set_ylabel(f'β({label})')
    ax.set_title(label)
    ax.grid(True, alpha=0.3)

plt.suptitle('TasteNet: β variation by OD distance', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# β 통계
print('\nTasteNet β statistics (across OD distances):')
print(f'{"Feature":<22} {"Mean β":>10} {"Std β":>10} {"Min β":>10} {"Max β":>10}')
print('-' * 65)
for i, label in enumerate(FEATURE_LABELS):
    b = betas[:, i]
    print(f'{label:<22} {b.mean():>10.4f} {b.std():>10.4f} {b.min():>10.4f} {b.max():>10.4f}')

In [ ]:
# Cell 12: ResLogit 잔차 분석

# 잔차 비중 (전체 test set)
res_ratio_total = None
mnl_mags, res_mags = [], []

reslogit_model.eval()
with torch.no_grad():
    for X, z, y, mask, w in test_loader:
        X, mask = X.to(DEVICE), mask.to(DEVICE)
        ratio = reslogit_model.get_residual_ratio(X, mask)
        mnl_mags.append(ratio['mnl_magnitude'])
        res_mags.append(ratio['residual_magnitude'])

avg_mnl = np.mean(mnl_mags)
avg_res = np.mean(res_mags)
avg_ratio = avg_res / (avg_mnl + avg_res + 1e-8)

print('=== ResLogit Residual Analysis ===')
print(f'  MNL component |V_MNL|:      {avg_mnl:.4f}')
print(f'  Residual component |V_res|: {avg_res:.4f}')
print(f'  Residual ratio:             {avg_ratio:.4f} ({avg_ratio*100:.1f}%)')

# β_MNL 변화 확인
print('\n=== ResLogit β_MNL (학습 후 vs 초기화) ===')
learned_beta = reslogit_model.beta_mnl.detach().cpu().numpy()
print(f'{"Feature":<22} {"Init (MNL)":>12} {"Learned":>12} {"Diff":>12}')
print('-' * 60)
for i, label in enumerate(FEATURE_LABELS):
    init_b = mnl_beta_scaled[i]
    learn_b = learned_beta[i]
    print(f'{label:<22} {init_b:>12.6f} {learn_b:>12.6f} {learn_b - init_b:>+12.6f}')

In [ ]:
# Cell 13: Training Curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

histories = [
    ('DNN', dnn_history),
    ('TasteNet', tn_history),
    ('ResLogit', rl_history),
]

for ax, (name, hist) in zip(axes, histories):
    epochs = range(1, len(hist['train_loss']) + 1)
    ax.plot(epochs, [l / 1e6 for l in hist['train_loss']], 'b-', label='Train', alpha=0.7)
    ax.plot(epochs, [l / 1e6 for l in hist['test_loss']], 'r-', label='Test', alpha=0.7)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (×10⁶)')
    ax.set_title(f'{name} Training Curve')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 14: 결과 저장
output = {
    'models': {},
    'comparison_summary': {},
}

for name, m in results.items():
    output['models'][name] = {
        'rho_sq': float(m['rho_sq']),
        'fpr_top1': float(m['fpr_top1']),
        'fpr_top3': float(m['fpr_top3']),
        'rmse': float(m['rmse']),
        'll_beta': float(m['ll_beta']),
    }

# Best model
best_model = max(results.items(), key=lambda x: x[1]['rho_sq'])
output['comparison_summary'] = {
    'best_model_rho_sq': best_model[0],
    'best_model_fpr': max(results.items(), key=lambda x: x[1]['fpr_top1'])[0],
    'residual_ratio': float(avg_ratio),
}

# TasteNet β range
output['tastenet_beta_range'] = {}
for i, label in enumerate(FEATURE_LABELS):
    b = betas[:, i]
    output['tastenet_beta_range'][label] = {
        'mean': float(b.mean()),
        'std': float(b.std()),
        'min': float(b.min()),
        'max': float(b.max()),
    }

out_path = Path(DATA_DIR) / 'dl_model_results.json'
with open(out_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, indent=2, ensure_ascii=False)
print(f'Results saved: {out_path}')

# Model weights 저장
torch.save(dnn_state, Path(DATA_DIR) / 'dnn_model.pt')
torch.save(tn_state, Path(DATA_DIR) / 'tastenet_model.pt')
torch.save(rl_state, Path(DATA_DIR) / 'reslogit_model.pt')
print('Model weights saved.')

In [ ]:
# Cell 15: 최종 비교 테이블 (논문/PPT용)
print('\n' + '=' * 80)
print('FINAL COMPARISON TABLE')
print('=' * 80)

# DataFrame 형태로 정리
comp_df = pd.DataFrame({
    'Model': ['MNL', 'DNN', 'TasteNet', 'ResLogit'],
    'McFadden ρ²': [results[m]['rho_sq'] for m in ['MNL', 'DNN', 'TasteNet', 'ResLogit']],
    'FPR Top-1': [results[m]['fpr_top1'] for m in ['MNL', 'DNN', 'TasteNet', 'ResLogit']],
    'FPR Top-3': [results[m]['fpr_top3'] for m in ['MNL', 'DNN', 'TasteNet', 'ResLogit']],
    'RMSE': [results[m]['rmse'] for m in ['MNL', 'DNN', 'TasteNet', 'ResLogit']],
    'Interpretable': ['Yes', 'No', 'Yes', 'Partial'],
}).set_index('Model')

print(comp_df.to_string(float_format=lambda x: f'{x:.4f}'))
print()
print(f'Best ρ²:  {best_model[0]} ({best_model[1]["rho_sq"]:.4f})')
print(f'ResLogit residual ratio: {avg_ratio:.1%}')